# Double DQN vs. DQN — CartPole-v1 y LunarLander-v3
**Implementación, experimentos y resultados · Tier 3 (opción B: mejoras de DQN)**

Este notebook explica Double DQN, lo compara con DQN usando **exactamente los mismos hiperparámetros**, y mide directamente la sobreestimación de los valores Q. Las gráficas leen los resultados ya obtenidos en `results/`; los entrenamientos completos no se ejecutan automáticamente.

**Ruta de exposición:** el problema de la sobreestimación → cómo lo corrige Double DQN → implementación → resultados en CartPole → resultados en LunarLander → conclusiones.

### Part 1 - Setup and Configurations
**¿Qué comparamos?** Dos agentes que solo se diferencian en **cómo calculan el objetivo TD**. Todo lo demás (red, buffer, exploración, hiperparámetros, semillas) es igual, así que cualquier diferencia se debe a Double DQN. Usamos el entorno del Tier 2 (LunarLander), como pide la opción B, y también CartPole, donde DQN sobreestimaba mucho.

In [ ]:
# ============================================================
# PART 1 - CONFIGURACIÓN
# ============================================================

import copy
import csv
import json
import time
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# Carpeta donde están los resultados de los entrenamientos.
# El notebook puede estar en la raíz del proyecto o en una subcarpeta.
RESULTS = Path("results") if Path("results").exists() else Path("../results")

SEMILLAS = [42, 123, 456]

# Protocolo común acordado con Q-learning
PROTOCOLO = {
    "eval_every": 10_000,           # evaluación periódica cada 10.000 pasos
    "eval_episodes": 10,            # ... con 10 episodios greedy
    "eval_seed_start": 10_000,      # semillas 10000-10009
    "final_eval_episodes": 100,     # evaluación final: 100 episodios
    "final_eval_seed_start": 20_000,  # semillas 20000-20099
}

CONFIG_CARTPOLE = {
    "exp_name": "cartpole_dqn", "env_id": "CartPole-v1",
    "total_steps": 300_000, "success_threshold": 475, **PROTOCOLO,
    "gamma": 0.99, "lr": 1e-4, "batch_size": 128, "buffer_size": 100_000,
    "hidden": [128, 128], "learning_starts": 1_000, "train_freq": 1,
    "target_update_freq": 1_000, "eps_start": 1.0, "eps_end": 0.05,
    "exploration_fraction": 0.1, "max_grad_norm": 10.0, "double": False,
}
CONFIG_LUNAR = {
    "exp_name": "lunarlander_dqn", "env_id": "LunarLander-v3",
    "total_steps": 500_000, "success_threshold": 200, **PROTOCOLO,
    "gamma": 0.99, "lr": 5e-4, "batch_size": 128, "buffer_size": 100_000,
    "hidden": [256, 256], "learning_starts": 10_000, "train_freq": 4,
    "target_update_freq": 1_000, "eps_start": 1.0, "eps_end": 0.05,
    "exploration_fraction": 0.2, "max_grad_norm": 10.0, "double": False,
}
# Double DQN: misma configuración, solo cambia `double`
CONFIG_CARTPOLE_DDQN = {**CONFIG_CARTPOLE, "exp_name": "cartpole_ddqn", "double": True}
CONFIG_LUNAR_DDQN = {**CONFIG_LUNAR, "exp_name": "lunarlander_ddqn", "double": True}

cambios = {k for k in CONFIG_LUNAR if CONFIG_LUNAR[k] != CONFIG_LUNAR_DDQN[k]}
print("Diferencias entre DQN y Double DQN:", cambios)

### Part 2 - The Problem: Maximization Bias
**¿Por qué DQN sobreestima?** El objetivo de DQN usa $\max_{a'} Q(s',a')$. Si las estimaciones tienen ruido, el máximo tiende a escoger la que **por azar** quedó más alta, así que en promedio el máximo de valores ruidosos es **mayor** que el valor verdadero, aunque cada estimación por separado no tenga sesgo.

**Experimento simple:** supongamos acciones cuyo valor verdadero es 0 y estimaciones con ruido normal. Comparamos el estimador de DQN (elegir y evaluar con las mismas estimaciones) con el de Double DQN (elegir con un conjunto y evaluar con otro independiente).

In [ ]:
# ============================================================
# PART 2 - DEMOSTRACIÓN DEL SESGO DE MAXIMIZACIÓN
# ============================================================

rng = np.random.default_rng(0)
REPETICIONES = 100_000

print(f"{'Acciones':<10}{'Valor real':<12}{'DQN: max':<12}{'Double DQN':<12}")
for n_acciones in [2, 4, 10]:
    est_online = rng.normal(0, 1, size=(REPETICIONES, n_acciones))   # red online (ruidosa)
    est_target = rng.normal(0, 1, size=(REPETICIONES, n_acciones))   # red objetivo (otro ruido)

    dqn = est_online.max(axis=1)                                     # elige y evalúa con lo mismo
    elegida = est_online.argmax(axis=1)                              # Double: elige con online...
    ddqn = est_target[np.arange(REPETICIONES), elegida]              # ...y evalúa con la objetivo

    print(f"{n_acciones:<10}{0.0:<12.2f}{dqn.mean():<12.3f}{ddqn.mean():<12.3f}")

print("\nCon más acciones el sesgo de DQN crece; el de Double DQN se mantiene cerca de 0.")

### Part 3 - The Fix: Double DQN Target
**¿Qué cambia?** Solo una línea del objetivo TD. Se separan las dos tareas del máximo:

- **DQN:** $y = r + \gamma\,(1-d)\,\max_{a'} Q_{\text{objetivo}}(s', a')$
- **Double DQN:** $y = r + \gamma\,(1-d)\,Q_{\text{objetivo}}\big(s',\ \arg\max_{a'} Q_{\text{online}}(s', a')\big)$

La red **online elige** la acción y la red **objetivo la evalúa**. Como sus errores son distintos, es menos probable que el error que hizo elegir una acción también infle su valor.

In [ ]:
# ============================================================
# RED Q
# ============================================================

class QNetwork(nn.Module):
    """MLP que recibe el estado y devuelve un valor Q por cada acción."""

    def __init__(self, obs_dim, n_actions, hidden=(128, 128)):
        super().__init__()
        capas, entrada = [], obs_dim
        for h in hidden:
            capas += [nn.Linear(entrada, h), nn.ReLU()]
            entrada = h
        capas.append(nn.Linear(entrada, n_actions))   # una salida por acción
        self.net = nn.Sequential(*capas)

    def forward(self, x):
        return self.net(x)


# Comprobación: 5 estados de CartPole -> 5 x 2 valores Q
red = QNetwork(obs_dim=4, n_actions=2)
salida = red(torch.zeros(5, 4))
print("Forma de la salida:", tuple(salida.shape))
print("Parámetros entrenables:", sum(p.numel() for p in red.parameters()))
assert salida.shape == (5, 2)


In [ ]:
# ============================================================
# PART 3 - CÁLCULO DEL OBJETIVO EN LAS DOS VARIANTES
# ============================================================

def objetivo_td(q_online, q_target, rewards, next_obs, dones, gamma, double):
    with torch.no_grad():
        if double:
            a_star = q_online(next_obs).argmax(dim=1, keepdim=True)      # elige la online
            next_q = q_target(next_obs).gather(1, a_star).squeeze(1)      # evalúa la objetivo
        else:
            next_q = q_target(next_obs).max(dim=1).values                 # max de la objetivo
        return rewards + gamma * (1.0 - dones) * next_q


# Comprobación: con redes distintas, el objetivo de Double DQN nunca supera al de DQN
torch.manual_seed(0)
online, target = QNetwork(4, 2), QNetwork(4, 2)
s2 = torch.randn(1000, 4)
r, d = torch.ones(1000), torch.zeros(1000)
y_dqn = objetivo_td(online, target, r, s2, d, 0.99, double=False)
y_ddqn = objetivo_td(online, target, r, s2, d, 0.99, double=True)
print(f"Objetivo medio DQN:        {y_dqn.mean():.4f}")
print(f"Objetivo medio Double DQN: {y_ddqn.mean():.4f}")
assert torch.all(y_ddqn <= y_dqn + 1e-6)   # Q_obj(s', a*) <= max_a Q_obj(s', a)

### Part 4 - Agent Implementation
**¿Qué código usamos?** El mismo agente del Tier 1 (red, buffer, epsilon-greedy, Huber loss, red objetivo). El parámetro `double` activa el objetivo de la Part 3 dentro de `update()`. La red Q ya se definió en la Part 3.

In [ ]:
# ============================================================
# REPLAY BUFFER
# ============================================================

class ReplayBuffer:
    """Memoria circular: cuando se llena, sobrescribe lo más antiguo."""

    def __init__(self, capacity, obs_dim, rng):
        self.capacity, self.rng = capacity, rng
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.dones = np.zeros(capacity, dtype=np.float32)   # 1 solo si terminated
        self.pos, self.size = 0, 0

    def add(self, obs, action, reward, next_obs, terminated):
        i = self.pos
        self.obs[i], self.next_obs[i] = obs, next_obs
        self.actions[i], self.rewards[i] = action, reward
        self.dones[i] = float(terminated)
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        # Muestreo uniforme: rompe la correlación entre pasos consecutivos
        idx = self.rng.integers(0, self.size, size=batch_size)
        return (torch.as_tensor(self.obs[idx]), torch.as_tensor(self.actions[idx]),
                torch.as_tensor(self.rewards[idx]), torch.as_tensor(self.next_obs[idx]),
                torch.as_tensor(self.dones[idx]))


# Comprobación con un buffer pequeño
buf = ReplayBuffer(capacity=3, obs_dim=4, rng=np.random.default_rng(0))
for k in range(5):                      # 5 transiciones en un buffer de 3
    buf.add(np.full(4, k), k % 2, 1.0, np.full(4, k + 1), terminated=(k == 4))
print("Tamaño:", buf.size, "| posiciones guardadas:", buf.obs[:, 0])
print("Se conservan las 3 transiciones más recientes (2, 3 y 4).")


In [ ]:
# ============================================================
# EXPLORACIÓN EPSILON-GREEDY CON DECAIMIENTO LINEAL
# ============================================================

def linear_epsilon(step, eps_start, eps_end, decay_steps):
    """Baja linealmente de eps_start a eps_end y luego se mantiene constante."""
    frac = min(1.0, step / max(1, decay_steps))
    return eps_start + frac * (eps_end - eps_start)


for paso in [0, 15_000, 30_000, 100_000, 300_000]:
    print(f"Paso {paso:>7}: epsilon = {linear_epsilon(paso, 1.0, 0.05, 30_000):.3f}")


In [ ]:
# ============================================================
# AGENTE DQN (incluye la opción Double DQN, usada en el Tier 3)
# ============================================================

class DQNAgent:
    def __init__(self, obs_dim, n_actions, cfg, rng):
        self.cfg, self.n_actions, self.rng = cfg, n_actions, rng
        self.gamma, self.batch_size = cfg["gamma"], cfg["batch_size"]
        self.double = cfg.get("double", False)

        self.q_net = QNetwork(obs_dim, n_actions, tuple(cfg["hidden"]))
        # Red objetivo: copia congelada que se sincroniza cada C pasos
        self.target_net = copy.deepcopy(self.q_net)
        self.target_net.requires_grad_(False)

        self.optimizer = torch.optim.Adam(self.q_net.parameters(), lr=cfg["lr"])
        self.buffer = ReplayBuffer(cfg["buffer_size"], obs_dim, rng)

    @torch.no_grad()
    def greedy_action(self, obs):
        q = self.q_net(torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0))
        return int(q.argmax(dim=1).item())

    def act(self, obs, epsilon):
        if self.rng.random() < epsilon:             # explorar
            return int(self.rng.integers(self.n_actions))
        return self.greedy_action(obs)              # explotar

    @torch.no_grad()
    def max_q(self, obs):
        """max_a Q(s, a): lo que la red cree que vale el estado s."""
        q = self.q_net(torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0))
        return float(q.max().item())

    def update(self):
        obs, actions, rewards, next_obs, dones = self.buffer.sample(self.batch_size)

        # Q(s, a) de las acciones que realmente se tomaron
        q_values = self.q_net(obs).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            if self.double:
                # Double DQN: la red ONLINE elige la acción, la red OBJETIVO la evalúa
                next_actions = self.q_net(next_obs).argmax(dim=1, keepdim=True)
                next_q = self.target_net(next_obs).gather(1, next_actions).squeeze(1)
            else:
                # DQN: la red objetivo elige y evalúa con el mismo max
                next_q = self.target_net(next_obs).max(dim=1).values
            # (1 - done) corta el bootstrap solo en estados terminales reales
            targets = rewards + self.gamma * (1.0 - dones) * next_q

        loss = F.smooth_l1_loss(q_values, targets)   # Huber loss
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), self.cfg.get("max_grad_norm", 10.0))
        self.optimizer.step()
        return float(loss.item()), float(q_values.mean().item())

    def sync_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())


# Comprobación: una actualización con datos aleatorios
_cfg = {"gamma": 0.99, "batch_size": 8, "hidden": [16, 16], "lr": 1e-3, "buffer_size": 50}
_ag = DQNAgent(4, 2, _cfg, np.random.default_rng(0))
for _ in range(20):
    _ag.buffer.add(np.random.randn(4), np.random.randint(2), 1.0, np.random.randn(4), False)
print("Loss de una actualización: %.4f" % _ag.update()[0])


In [ ]:
# ============================================================
# EVALUACIÓN GREEDY (EPSILON = 0)
# ============================================================

def evaluate(agent, env, seeds, gamma):
    """Un episodio greedy por semilla. También mide la sobreestimación de Q en s0."""
    returns, q0_est, q0_real = [], [], []
    for s in seeds:
        obs, _ = env.reset(seed=int(s))
        q0_est.append(agent.max_q(obs))          # lo que la red predice para s0
        done, ret, disc, factor = False, 0.0, 0.0, 1.0
        while not done:
            obs, r, terminated, truncated, _ = env.step(agent.greedy_action(obs))
            ret += r
            disc += factor * r                   # retorno descontado real desde s0
            factor *= gamma
            done = terminated or truncated
        returns.append(ret)
        q0_real.append(disc)
    return np.array(returns), np.array(q0_est), np.array(q0_real)


### Part 5 - Training
**¿Cómo entrenamos?** Con la misma función por pasos que en el Tier 1. Las 12 corridas (2 entornos × 2 métodos × 3 semillas) **no se ejecutan automáticamente**. Para una demostración corta, cambia `DEMO = True`.

In [ ]:
# ============================================================
# ENTRENAMIENTO POR PASOS (mismo código que experiments/train_dqn.py)
# ============================================================

def entrenar_dqn(cfg, seed, results_dir=RESULTS):
    out = Path(results_dir) / cfg["exp_name"] / f"seed_{seed}"
    out.mkdir(parents=True, exist_ok=True)

    torch.set_num_threads(1)
    np.random.seed(seed)
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    env, eval_env = gym.make(cfg["env_id"]), gym.make(cfg["env_id"])
    env.action_space.seed(seed)
    obs, _ = env.reset(seed=seed)
    agent = DQNAgent(env.observation_space.shape[0], env.action_space.n, cfg, rng)

    total = cfg["total_steps"]
    decay = int(cfg["exploration_fraction"] * total)
    eval_seeds = range(cfg["eval_seed_start"], cfg["eval_seed_start"] + cfg["eval_episodes"])
    final_seeds = range(cfg["final_eval_seed_start"],
                        cfg["final_eval_seed_start"] + cfg["final_eval_episodes"])

    filas, train_time, t0 = [], 0.0, time.perf_counter()
    steps_to_threshold = None
    for step in range(1, total + 1):
        eps = linear_epsilon(step, cfg["eps_start"], cfg["eps_end"], decay)
        action = agent.act(obs, eps)
        next_obs, r, terminated, truncated, _ = env.step(action)
        # Se guarda `terminated`, NO `truncated`: cortar por tiempo no es un fallo
        agent.buffer.add(obs, action, r, next_obs, terminated)
        obs = next_obs
        if terminated or truncated:
            obs, _ = env.reset()

        if step > cfg["learning_starts"] and step % cfg["train_freq"] == 0:
            agent.update()
        if step % cfg["target_update_freq"] == 0:
            agent.sync_target()

        if step % cfg["eval_every"] == 0:          # se pausa el cronómetro
            train_time += time.perf_counter() - t0
            ret, q0e, q0r = evaluate(agent, eval_env, eval_seeds, cfg["gamma"])
            filas.append([step, ret.mean(), ret.std(), eps, train_time, q0e.mean(), q0r.mean()])
            if steps_to_threshold is None and ret.mean() >= cfg["success_threshold"]:
                steps_to_threshold = step
            print(f"seed {seed} | paso {step:>7} | eval {ret.mean():7.2f} | eps {eps:.3f}")
            t0 = time.perf_counter()
    train_time += time.perf_counter() - t0

    pd.DataFrame(filas, columns=["step", "eval_mean", "eval_std", "epsilon", "train_time_s",
                                 "q0_estimate", "q0_actual"]).to_csv(out / "eval.csv", index=False)
    ret, q0e, q0r = evaluate(agent, eval_env, final_seeds, cfg["gamma"])
    resumen = {"seed": seed, "final_mean": float(ret.mean()), "final_std": float(ret.std()),
               "solved": bool(ret.mean() >= cfg["success_threshold"]),
               "steps_to_threshold": steps_to_threshold, "train_time_s": round(train_time, 2),
               "final_q0_estimate": float(q0e.mean()), "final_q0_actual": float(q0r.mean()),
               "final_returns": ret.tolist(), "best_ckpt_mean": float("nan")}
    json.dump(resumen, open(out / "final.json", "w"), indent=2)
    torch.save(agent.q_net.state_dict(), out / "model_final.pt")
    return resumen


In [ ]:
DEMO = False

if DEMO:
    demo_cfg = {**CONFIG_CARTPOLE_DDQN, "exp_name": "demo_ddqn", "total_steps": 20_000,
                "eval_every": 5_000, "final_eval_episodes": 10}
    print(entrenar_dqn(demo_cfg, seed=42, results_dir="/tmp/demo_results"))
else:
    print("Demostración desactivada. Entrenamientos completos:")
    print("  python -m experiments.run_seeds --config experiments/configs/lunarlander_ddqn.json")
    print("  python -m experiments.run_seeds --config experiments/configs/cartpole_dqn.json "
          "--set double=true --set exp_name=cartpole_ddqn")

In [ ]:
# ============================================================
# FUNCIONES PARA LEER Y GRAFICAR RESULTADOS
# ============================================================

COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]


def cargar_finales(exp):
    """Tabla con la evaluación final (100 episodios) de cada semilla."""
    filas = []
    for s in SEMILLAS:
        f = RESULTS / exp / f"seed_{s}" / "final.json"
        if f.exists():
            d = json.load(open(f))
            filas.append({"semilla": s, "media": d["final_mean"], "std": d["final_std"],
                          "resuelto": d["solved"], "pasos_umbral": d["steps_to_threshold"],
                          "tiempo_min": d["train_time_s"] / 60})
    return pd.DataFrame(filas)


def cargar_curvas(exp, columna="eval_mean"):
    """Matriz (semillas x puntos de evaluación) alineada por paso."""
    series = []
    for s in SEMILLAS:
        f = RESULTS / exp / f"seed_{s}" / "eval.csv"
        if f.exists():
            df = pd.read_csv(f)
            if columna in df:
                series.append(df.set_index("step")[columna].rename(s))
    if not series:
        return None, None
    tabla = pd.concat(series, axis=1).dropna()
    return tabla.index.values, tabla.values.T


def resumen(exp, nombre):
    df = cargar_finales(exp)
    if df.empty:
        return None
    return {"método": nombre,
            "media entre semillas": round(df["media"].mean(), 2),
            "std entre semillas": round(df["media"].std(ddof=0), 2),
            "por semilla (42/123/456)": " / ".join(f"{m:.1f}" for m in df["media"]),
            "resueltas": f"{int(df['resuelto'].sum())}/{len(df)}",
            "pasos al umbral": " / ".join("no" if pd.isna(p) else f"{int(p)//1000}k"
                                          for p in df["pasos_umbral"]),
            "tiempo (min)": round(df["tiempo_min"].mean(), 1)}


def graficar(exps, etiquetas, umbral=None, clip=None, columna="eval_mean",
             titulo="", ylabel="Retorno de evaluación (10 episodios greedy)"):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    for i, (exp, lab) in enumerate(zip(exps, etiquetas)):
        x, Y = cargar_curvas(exp, columna)
        if x is None:
            print(f"(sin datos para {exp})")
            continue
        m, s = Y.mean(axis=0), Y.std(axis=0)
        sup = m + s if clip is None else np.minimum(m + s, clip)
        ax.plot(x, m, color=COLORES[i], lw=2, label=f"{lab} (n={len(Y)})")
        ax.fill_between(x, m - s, sup, color=COLORES[i], alpha=0.18, lw=0)
    if umbral is not None:
        ax.axhline(umbral, color="gray", ls=":", lw=1, label=f"Umbral: {umbral}")
    ax.set_xlabel("Pasos del entorno")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:.0f}k"))
    ax.set_ylabel(ylabel)
    ax.set_title(titulo)
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    plt.show()


def graficar_sobreestimacion(exps, etiquetas, titulo=""):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    for i, (exp, lab) in enumerate(zip(exps, etiquetas)):
        x, est = cargar_curvas(exp, "q0_estimate")
        _, real = cargar_curvas(exp, "q0_actual")
        if x is None:
            print(f"(sin datos para {exp})")
            continue
        ax.plot(x, est.mean(axis=0), color=COLORES[i], ls="--", lw=2, label=f"{lab}: Q estimado")
        ax.plot(x, real.mean(axis=0), color=COLORES[i], lw=2, label=f"{lab}: retorno real")
    ax.set_xlabel("Pasos del entorno")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:.0f}k"))
    ax.set_ylabel("Valor del estado inicial")
    ax.set_title(titulo)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


### Part 6 - Results on CartPole-v1
**¿Qué obtuvimos?** Double DQN fue más estable y más rápido. La semilla 42, que con DQN colapsaba a 108.5, terminó en 421.2. La gráfica de sobreestimación muestra por qué: el valor real máximo es $1/(1-\gamma) = 100$, y DQN termina estimando ≈109.5, mientras Double DQN se queda en ≈106.

In [ ]:
# ============================================================
# PART 6 - CARTPOLE: DQN VS DOUBLE DQN
# ============================================================

tabla = [r for r in [resumen("cartpole_dqn", "DQN"), resumen("cartpole_ddqn", "Double DQN")] if r]
if tabla:
    display(pd.DataFrame(tabla).set_index("método").T)

graficar(["cartpole_dqn", "cartpole_ddqn"], ["DQN", "Double DQN"], umbral=475, clip=500,
         titulo="CartPole-v1: DQN vs. Double DQN")
graficar_sobreestimacion(["cartpole_dqn", "cartpole_ddqn"], ["DQN", "Double DQN"],
                         titulo="CartPole-v1: Q estimado (--) vs. retorno real (—) en s0")

### Part 7 - Results on LunarLander-v3
**¿Qué obtuvimos?** Aquí Double DQN **no cambió significativamente** el resultado final: la diferencia de unos 10 puntos es mucho menor que la variación entre episodios (≈69). Aprendió antes entre 80k y 180k pasos, pero luego las curvas se juntan. La razón: en LunarLander DQN casi no sobreestimaba.

In [ ]:
# ============================================================
# PART 7 - LUNARLANDER: DQN VS DOUBLE DQN
# ============================================================

tabla = [r for r in [resumen("lunarlander_dqn", "DQN"),
                     resumen("lunarlander_ddqn", "Double DQN")] if r]
if tabla:
    display(pd.DataFrame(tabla).set_index("método").T)

graficar(["lunarlander_dqn", "lunarlander_ddqn"], ["DQN", "Double DQN"], umbral=200,
         titulo="LunarLander-v3: DQN vs. Double DQN")
graficar_sobreestimacion(["lunarlander_dqn", "lunarlander_ddqn"], ["DQN", "Double DQN"],
                         titulo="LunarLander-v3: Q estimado (--) vs. retorno real (—) en s0")

### Part 8 - Summary Across Both Environments

In [ ]:
# ============================================================
# PART 8 - RESUMEN (valores registrados de las corridas completas)
# ============================================================

resumen_tier3 = pd.DataFrame([
    {"entorno": "CartPole", "método": "DQN", "media final": "369.5 ± 184.5", "resueltas": "2/3",
     "pasos al umbral": "170k–200k", "Q(s0) estimado / real": "109.5 / 87.8"},
    {"entorno": "CartPole", "método": "Double DQN", "media final": "473.8 ± 37.1", "resueltas": "2/3",
     "pasos al umbral": "120k–130k", "Q(s0) estimado / real": "106.0 / 99.0"},
    {"entorno": "LunarLander", "método": "DQN", "media final": "250.2 ± 9.3", "resueltas": "3/3",
     "pasos al umbral": "190k–270k", "Q(s0) estimado / real": "71.3 / 71.3"},
    {"entorno": "LunarLander", "método": "Double DQN", "media final": "240.8 ± 28.9", "resueltas": "3/3",
     "pasos al umbral": "90k–260k", "Q(s0) estimado / real": "71.3 / 69.4"},
])
resumen_tier3

### Part 9 - Conclusions

**Conclusiones para defender:**

- Double DQN **corrige el sesgo de maximización**: elige la acción con la red online y la evalúa con la red objetivo (Part 2 y Part 3).
- **CartPole:** DQN sobreestimaba (≈109.5 cuando el máximo posible es 100). Double DQN redujo la variación entre semillas de 184.5 a 37.1 y llegó al umbral antes (120k–130k vs. 170k–200k pasos). Reduce el sesgo, pero no lo elimina (≈106).
- **LunarLander:** DQN casi no sobreestimaba al final (71.3 vs. 71.3), y Double DQN no cambió el resultado de forma significativa.
- **Idea principal:** Double DQN ayuda **en la medida en que haya sobreestimación**. En CartPole, con 2 acciones de valor muy parecido y recompensa constante, el máximo sobre estimaciones ruidosas está más sesgado.
- **Limitación:** con 3 semillas, las diferencias pequeñas (como los 10 puntos en LunarLander) no son concluyentes; las 2 de 3 semillas resueltas en CartPole son iguales para ambos métodos.
- **Conexión con métodos continuos:** TD3 usa la misma idea (el mínimo de dos redes Q) para combatir la sobreestimación en acciones continuas.